# **1. Import Statements**

In [ ]:
%%capture
!pip install -q --no-deps scikeras

In [ ]:
%%time
%matplotlib inline

## Working with system!
import os
import sys
import subprocess
import multiprocessing
import warnings
import time
import copy

## most common libraries
import numpy
import numpy as np
import pandas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
import seaborn as sns
import math

## Libraries to Build Pipeline for Data-Transforms
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

# mathStuff
import math
from scipy.stats import skew, kurtosis
from scipy.stats import chi2_contingency

# sklearn MODELS
import sklearn
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn import svm
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.decomposition import PCA

# Other awesome models
from xgboost import XGBClassifier
import xgboost as xgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import AdaBoostClassifier

## Handeling output!
from IPython.display import clear_output
import tqdm

## exceptionals
import sqlite3
import PIL

warnings.filterwarnings('ignore')
print("\033[1mImported Important Libraries\033[0m")

#  **2. Data Overview**

## **2.1 Loading Data && Data Description**

In [ ]:
# changing 'os' path to '/kaggle/input'
os.chdir('/kaggle/input/')
inputPTH = os.path.join(os.getcwd(), 'hr-analytics-employee-attrition-and-performance')

print("Different Tables from the Dataset are:")
[print("\t",  i) for i in os.listdir(inputPTH) if ".csv" in i];


# Loading data!
'''The Main Data Source'''
performanceAnalysis = pandas.read_csv(os.path.join(inputPTH, 'PerformanceRating.csv'))
employeeData = pandas.read_csv(os.path.join(inputPTH, 'Employee.csv'))

ratingAsForeinKey = pandas.read_csv(os.path.join(inputPTH, 'RatingLevel.csv'))
satisfactionAsForeinKey = pandas.read_csv(os.path.join(inputPTH, 'SatisfiedLevel.csv'))
educationAsForeinKey = pandas.read_csv(os.path.join(inputPTH, 'EducationLevel.csv'))

# os.system('cd hr-analytics-employee-attrition-and-performance && cat DimDate.txt');

print("\nData loaded to RAM!\n")

print(f'Exploring Table \033[1m"PerformanceRating"\033[0m:',
        f'\t\033[1mTotal Features:\033[0m {performanceAnalysis.shape[1]}',
        f'\t\033[1mTotal Rows:\033[0m {performanceAnalysis.shape[0]}', 
        f'\t\033[1mFeatures Include:\033[0m {list(performanceAnalysis.columns)}', sep="\n", end="\n\n")

print(f'Exploring Table \033[1m"Employee"\033[0m :',
        f'\t\033[1mTotal Features:\033[0m {employeeData.shape[1]}',
        f'\t\033[1mTotal Rows:\033[0m {employeeData.shape[0]}',
        f'\t\033[1mFeatures Include:\033[0m {list(employeeData.columns)}', sep="\n")

## **2.2 Feature Overview**

> **The Tables: "*RatingLevel.csv*", "*SatisfiedLevel.csv*", "*EducationLevel.csv*" from the dataset are "lookup table"**

>  **Lookup table are used to map categorical values to corresponding numeric codes (ex- mapping [Low, Medium, High] to [0, 1, 2])**

> **Other Tables including "*PerformanceRating.csv*", "*Employee.csv*", that the one we'll be focus on..**

In [ ]:
print("\033[1;7m  Table: Performance Rating  ")
performanceAnalysis.head().T

In [ ]:
print(f"\033[1;7m  Table: Employee Data  ")
employeeData.head().T

In [ ]:
# From Above Observation
categoricalFeatures = [
    'EnvironmentSatisfaction',
    'JobSatisfaction', 
    'RelationshipSatisfaction', 
    'TrainingOpportunitiesWithinYear', 
    'TrainingOpportunitiesTaken', 
    'WorkLifeBalance', 
    'SelfRating', 
    'ManagerRating',

    'Gender',
    'BusinessTravel', 
    'Department',
    'State',
    'Ethnicity',
    'Education', 
    'EducationField', 
    'JobRole',
    'MaritalStatus',
    'StockOptionLevel',
    'OverTime',
    'Attrition',
]

continuousFeatures = [
    'Age',
    'DistanceFromHome (KM)',
    'Salary',
    
    'YearsAtCompany',
    'YearsInMostRecentRole',
    'YearsSinceLastPromotion',
    'YearsWithCurrManager',
]

print(f'\033[1mCategorical Features:\033[0m {categoricalFeatures}', end='\n\n')

print(f'\033[1mContinuous Features:\033[0m {continuousFeatures}', end='\n\n')

## **2.3 Target Variable**

In [ ]:
Target = ["Attrition"]
print(f'\033[1mTarget Variable: \033[0m{Target[0]}')

# **3. Data Preprocessing**

## **3.1 Merging Table based on Primary Key 'Employee ID'**

In [ ]:
train_df = pandas.merge(performanceAnalysis, employeeData, on="EmployeeID")
# train_df.columns
print("Data after merge: ")
train_df.head().T

## **3.2 Handling Missing Data**

In [ ]:
print(train_df.isnull().any())

print("\nThere is \033[1mNo Missing Values\033[0m in the Dataset")

## **3.3 Checking for Outlier**

In [ ]:
## handeling outliers
cont_FEATURES = ['Age', 'Salary']

def plot_outliers(data, target, train_df, feature, threshold=2.3):
    mean, std = numpy.mean(train_df), numpy.std(train_df)
    z_score = numpy.abs((train_df-mean) / std)
    good = z_score < threshold

    print(f"Rejection {(~good).sum()} points")
    visual_scatter = numpy.random.normal(size=train_df.size)
    if((~good).sum()>0):
        plt.scatter(train_df[good], visual_scatter[good], s=2, label="Good", color="#4CAF50")
        plt.scatter(train_df[~good], visual_scatter[~good], s=8, label="Bad", color="#F44336")
        plt.legend(loc='upper right')
        plt.title(feature)
        plt.show();
    return None, None
        

for feature in cont_FEATURES:
        
    print(feature, end=' : ')
    plot_outliers(train_df ,'', train_df[feature], feature)
    print()

## **3.4 Data Transformation**

<font size="4"><span style="color:red">**Data Transforms:**</span></font>

>  **1. Columns to Drop-off: [*PerformanceID* && *EmployeeID*]**

>  2. Extracting [Year / Month / Date] from DateTimeColumns*: [*HireDate* && *ReviewDate*]

>  3. [*FirstName*; *LastName*;] -> removing from dataset for now!

>  **4. Applying OneHotEncoding to : [*BusinessTravel*, *Department*, *Ethnicity*, *EducationField*, *JobRole*, *MaritalStatus*]**

>  **5. Applying Lable Encoding to: [*Gender*, *State*, *OverTime*]**

> **6. Applying Log transform to: [*Salary*]**

> **7. Handling Outliers in [*Age*] Using the IQR Method**

In [ ]:
%%capture
# CUSTOME CLASSES FOR SPECIFIC USECASES -> 
#     Dropping Multiple Columns, OneHotEncoding, LabelEncoding, HandelingOutlier;

# Class 1: Removing Specific/Multiple Columns
class RemoveColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_drop=None):
        self.columns_to_drop = columns_to_drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.columns_to_drop:
            X = X.drop(self.columns_to_drop, axis=1)
        return X

# Class 2: OneHotEncoding for Specific Columns
class OneHotEncodeColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_encode=None):
        self.columns_to_encode = columns_to_encode
        self.encoder = None

    def fit(self, X, y=None):
        if self.columns_to_encode:
            self.encoder = OneHotEncoder(drop='first', sparse=False)
            self.encoder.fit(X[self.columns_to_encode])
        return self

    def transform(self, X):
        X = X.copy()
        if self.columns_to_encode and self.encoder:
            encoded_cols = pd.DataFrame(
                self.encoder.transform(
                    X[self.columns_to_encode]
                ), 
                columns = self.encoder.get_feature_names_out(self.columns_to_encode), 
                index = X.index
            )
            
            X = X.drop(self.columns_to_encode, axis=1)
            X = pd.concat([X, encoded_cols], axis=1)
        return X

# Class 3: Handling Outliers for Specific Features
class HandleOutliers(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_check=None, method='IQR'):
        self.columns_to_check = columns_to_check
        self.method = method

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        
        # Use default IQR method if no specific method or columns are provided
        if self.columns_to_check:
            for col in self.columns_to_check:
                if self.method == 'LogTransform':
                    for col in self.columns_to_check:
                        X[col] = X[col].apply(lambda x: np.log1p(x) if x > 0 else 0)
                
                else:
                    Q1 = X[col].quantile(0.25)
                    Q3 = X[col].quantile(0.75)
                    IQR = Q3 - Q1
                    X = X[~((X[col] < (Q1 - 1.5 * IQR)) | (X[col] > (Q3 + 1.5 * IQR)))]
        return X
    
# Class 4: Label Encoding for Specific Columns
class LabelEncodeColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_encode=None):
        self.columns_to_encode = columns_to_encode
        self.encoders = {}

    def fit(self, X, y=None):
        # Initialize a LabelEncoder for each column and fit it
        for col in self.columns_to_encode:
            le = LabelEncoder()
            le.fit(X[col])
            self.encoders[col] = le
        return self

    def transform(self, X):
        X = X.copy()  # Avoid modifying the original DataFrame
        for col in self.columns_to_encode:
            X[col] = self.encoders[col].transform(X[col])
        return X


In [ ]:
# Specifying Columns for Differnt Cases!!
dropColumns = ['PerformanceID', 'EmployeeID', 'FirstName', 'LastName', "HireDate", "ReviewDate"]
encodeColumns = ['BusinessTravel', 'Department', 'Ethnicity', 'EducationField', 'JobRole', 'MaritalStatus']
labelEncodeColumns = ['Gender', 'State', 'OverTime']
checkOutliersColumns = ['Age', 'Salary']

# Defining the pipeline
pipeline = Pipeline(
    steps=[
        ('remove_columns', RemoveColumns(columns_to_drop=dropColumns)), 
        
        ('onehot_encode', OneHotEncodeColumns(columns_to_encode=encodeColumns)),
        
        ('label_encode', LabelEncodeColumns(columns_to_encode=labelEncodeColumns)),
        
        ('handle_outliers_Age', HandleOutliers(columns_to_check=["Age"], method='IQR')),
        
        ('handle_outliers_Salary', HandleOutliers(columns_to_check=["Salary"], method='LogTransform')),
])

train_df_transformed = pipeline.fit_transform(train_df.copy())

# removing the OneHotEncode Features from the List of Categorical Features
[categoricalFeatures.remove(x) if x in categoricalFeatures else None for x in encodeColumns];

# Specific Changes in Target Varibale 'Attrition'
train_df_transformed['Attrition'] = train_df_transformed['Attrition'].map({"Yes" : 1, "No" : 0})

In [ ]:
print("Changes after dataTransform in column 'Salary' (applied log transform!)")

pd.DataFrame({
    "Before": train_df.Salary.describe().drop('count'),
    "After": train_df_transformed.Salary.describe().drop('count')
})

In [ ]:
print("Changes after dataTransform in Column 'Age' (applied IQR transform!)") 
print("Very Slight Changes (applied to the Outliers!)")

pd.DataFrame({
    "Before": train_df.Age.describe().drop('count'),
    "After": train_df_transformed.Age.describe().drop('count')
})

In [ ]:
print("Value (Marketing) is repeating on itself, need to check (if it's a mistake) OR (intentional)")
train_df.EducationField.value_counts()

# **4. Exploratory Data Analysis**

## **4.1 Validity of Target Variable!**

In [ ]:
'''Attrition is generally considered a normal process as employees move to new phases of their lives or careers. (GOOGLE)'''

# Checking for Skewness/Kurtosis in the Target Variable i.e. 'Attrition'

plt.subplots(1, figsize=(2, 2))
train_df_transformed['Attrition'].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Distribution of Attrition')
plt.xlabel('Attrition')
plt.ylabel('Count')
plt.show()

# Skewness and Kurtosis
skewness = skew(train_df_transformed['Attrition'])
kurt = kurtosis(train_df_transformed['Attrition'])

print(f'Skewness of Attrition: {skewness}')
print(f'Kurtosis of Attrition: {kurt}')

print('''
\033[1mTarget Variable is Right-Skewed\033[0m
There are more employees who stayed within the company than those who left!
''')

## **4.2 Univariant Analysis <span style="color:gray"><font size="2">(Data Distribution Within A Feature)**</font></span>

In [ ]:
# Categorical Values:
print("Data Distribution for Categorical Feature:")

# Creating Grid for plotting
fig, axes = plt.subplots(nrows=3, ncols=5, figsize=(21, 8), squeeze=True)
axes = axes.flatten()

# Loop through the columns
for i, column in enumerate(categoricalFeatures):
    if column in train_df.columns: # using train_df instead of train_df_transformed!
        # Plot value counts for each column
        train_df[column].value_counts().plot(kind='bar', ax=axes[i], color='skyblue')
        axes[i].set_title(f'{column}', fontsize=10)
        axes[i].set_xlabel("", fontsize=0)
        axes[i].set_ylabel('Count', fontsize=9)
        axes[i].tick_params(axis='both', which='major', labelsize=9)

# Adjust layout
# plt.title("Categorical Features")
axes[14].axis('off')
plt.tight_layout()
plt.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()

In [ ]:
# Continuous Values:
print("Data Distribution for Continuous Feature:")

# print("\033[1mBEFORE: \033[0m")
# Creating Grid for plotting
# fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(18, 4.5), squeeze=True)

# # Loop through the columns
# for i, column in enumerate(continuousFeatures):
#     if column in train_df.columns:
#         # Plot histogram
#         sns.histplot(train_df[column].dropna(), bins=20, kde=True, ax=axes[i//4][i%4], color='skyblue')

# # Adjust layout
# plt.tight_layout()
# plt.show()

# print("AFTER: ")

# Creating Grid for plotting
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(18, 4.5), squeeze=True)

# Loop through the columns
for i, column in enumerate(continuousFeatures):
    if column in train_df_transformed.columns:
        # Plot histogram
        sns.histplot(train_df_transformed[column].dropna(), bins=20, kde=True, ax=axes[i//4][i%4], color='skyblue')

# Adjust layout
axes[1][3].axis('off')
plt.tight_layout()
plt.show()

## **4.3 Bivariate Analysis**  <span style="color:gray"><font size="2">**(used help from gpt for plotting)**</font></span>

In [ ]:
# Relation between Education and Job Role

countEduJob = pd.crosstab(employeeData['EducationField'], employeeData['JobRole'])

total_counts = countEduJob.sum(axis=1)
sorted_education_indices = total_counts.sort_values(ascending=False).index
sorted_job_roles = countEduJob.loc[sorted_education_indices].sum(axis=0).sort_values(ascending=False).index
sorted_contingency_table = countEduJob.loc[sorted_education_indices, sorted_job_roles]

ax = sorted_contingency_table.plot(kind='bar', stacked=True)

# Adjust layout
ax.legend(title='Job Role', loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')
plt.title('Relationship between Education Field and Job Role')
plt.xlabel('Education Field')
plt.ylabel('Count')
plt.xticks(rotation=85)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate proportions
crosstab = pd.crosstab(employeeData['JobRole'], employeeData['Attrition'])
proportions = crosstab.div(crosstab.sum(axis=1), axis=0)
proportions = proportions.sort_values(by='Yes', ascending=False)
plt.figure(figsize=(10, 6))
proportions.plot(kind='bar', stacked=True)

# Adjust legend
plt.title('Attrition by Job Role')
plt.xlabel('Job Role')
plt.ylabel('Proportion')
plt.xticks(rotation=85)
plt.legend(title='Attrition', loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')
plt.tight_layout()  # Adjust layout to make room for the legend
plt.show()

on going

## **4.4 Feature Correlation**

In [ ]:
# relation = train_df_transformed.drop(["HireDate", "ReviewDate"], axis=1).corr();
corr_features = train_df_transformed.columns

cc = np.corrcoef(train_df_transformed, rowvar = False)

plt.figure(figsize=(17, 11))
seaborn.heatmap(cc*10, # all value should be interpret as "values / 10" -> Hence values range for corr is: [-1, 1]
                center=0, 
                cmap='coolwarm', 
                annot=True, 
                fmt = '.0f',
                xticklabels=corr_features, 
                yticklabels=corr_features)


plt.title('Correlation Matrix', fontsize=20)
plt.show()

<font size="4"><span style="color:red">**Conclusions:**</span></font>

> **As the Target Variable is Right Skewed, we have to use [F1-score, AUC-ROC, Precision, Recall] while training the model to avoid [misleading performance evaluations]**

> **Lower values of <span style="color:green">[*Age*, *Salary*, *YearsSinceLastPromotion*]</span> may contribute to higher '*Attrition*' rates (indicating a negative correlation).**

> **<span style="color:green">[*YearsAtCompany*]</span> and <span style="color:green">[*Salary*]</span> seem to have a weak positive correlation (~0.2), which is expected as employees who stay longer tend to earn more.**

> **Most of the features, like <span style="color:green">[*Education*, *OverTime*, and *StockOptionLevel*]</span> have very weak correlations (close to 0). This suggests these features might not have significant linear relationships with others.**

> **Features like <span style="color:green">[*SelfRating*, *ManagerRating*]</span> have almost no correlation with Attrition or other key features, suggesting they might not be as important for predicting employee behavior or attrition in this dataset.**

# **5. Key Feature Analysis**

## 5.1 Feature Importance using Tree-Based Models

In [ ]:
# feature importance
# focusing on the most impactful features.

In [ ]:
# XGBOOST CLASSIFIER

# defining data (X and y)
X = train_df_transformed.drop('Attrition', axis=1)
y = train_df_transformed['Attrition']

# Calculate scale_pos_weight (since the target variable is imbalanced!)
neg_class = len(y[y == 0])  # Count of class 0 (non-attrition)
pos_class = len(y[y == 1])  # Count of class 1 (attrition)
scale_pos_weight = neg_class / pos_class

# fitting the XGBClassifier Model
model = xgb.XGBClassifier(scale_pos_weight=scale_pos_weight)
model.fit(X, y)

# Get feature importances
importances = model.feature_importances_
features = X.columns
feature_importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
top_features_df = feature_importance_df.head(7) # filtering the top 7 features!

# Plot the feature importance
# plt.style.use('dark_background')
plt.figure(figsize=(10, 6))
plt.barh(top_features_df['Feature'], top_features_df['Importance'], color='skyblue')

plt.xlabel('Feature Importance', fontsize=13)
plt.ylabel('Features', fontsize=13)
plt.title('Top 7 Most Important Features using XGBoost')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important feature on top
plt.show()

## 5.2 PCA (Principal Component Analysis)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Define data (X and y)
X = train_df_transformed.drop('Attrition', axis=1).copy()
y = train_df_transformed['Attrition'].copy()

# Standardize the features
X_scaled = StandardScaler().fit_transform(X)

# Apply PCA
pca = PCA(n_components=3)  # 3 principal components
X_pca = pca.fit_transform(X_scaled)

# Get the PCA components
components_ = pd.DataFrame(pca.components_.T, columns=[f'PC{i+1}' for i in range(3)], index=X.columns)

# For each principal component [get the top 3 contributing features]
top_features = {}
for pc in components_.columns:
    top_features[pc] = components_[pc].sort_values(ascending=False).head(3)

for pc, features in top_features.items():
    print(f"Top features for {pc}:\n{features}\n")

# **6. Modeling**

## 6.1 Training a Simple Neural Network

In [ ]:
# KerasClassifier
import keras
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import roc_auc_score
from scikeras.wrappers import KerasClassifier

# Split the data [train, test]
X_tr, X_va = train_test_split(train_df_transformed, test_size=0.2, stratify=train_df_transformed.Attrition, random_state=1)
y_tr = X_tr.pop('Attrition')
y_va = X_va.pop('Attrition')

# Keras model with two hidden layers
def get_model(meta):
    model = keras.models.Sequential()
    model.add(keras.layers.Input(meta["X_shape_"][1:]))
    model.add(keras.layers.Dense(128, kernel_initializer='lecun_normal', activation='relu'))
    model.add(keras.layers.Dense(64, kernel_initializer='lecun_normal', activation='relu'))
    model.add(keras.layers.Dense(1, kernel_initializer='lecun_normal', activation='sigmoid'))
    return model

In [ ]:
model1 = KerasClassifier(
        get_model,
        loss="binary_crossentropy",
        optimizer=keras.optimizers.AdamW(learning_rate=0.03),
        validation_split=0.03,
        batch_size=2048,
        epochs=20,
        callbacks=[
            # Learning rate decay when validation loss stops improving
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',  # The metric to monitor (typically validation loss)
                factor=0.5,          # Factor by which to reduce the learning rate
                patience=3,          # Number of epochs with no improvement after which learning rate will be reduced
                verbose=1,           # Verbosity mode, 1 = print messages
                min_lr=1e-6          # Lower bound on the learning rate
            ),
            # Early stopping to avoid overfitting
            keras.callbacks.EarlyStopping(
                monitor='val_loss',  # Metric to monitor
                patience=5,          # Number of epochs with no improvement to stop training
                verbose=1,           # Verbosity mode
                restore_best_weights=True  # Restore model weights from the epoch with the best value of the monitored metric
            )
        ]
    )

In [ ]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(y_tr), 
    y=y_tr
)

model1.fit(X_tr, y_tr, class_weight=dict(enumerate(class_weights)))

## 6.2 Evaluating Model Result

In [ ]:
print(f"# AUC: {roc_auc_score(y_va, model1.predict_proba(X_va)[:,1]):.5f}")

In [ ]:
from sklearn.metrics import classification_report, roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# Predict on validation data
y_pred = model.predict(X_va)
y_pred_proba = model.predict_proba(X_va)[:, 1]  # For ROC-AUC curve (binary classification)

# 1. Classification report (Precision, Recall, F1-Score, and Accuracy)
print(classification_report(y_va, y_pred))

# 2. Confusion matrix
conf_matrix = confusion_matrix(y_va, y_pred)
ConfusionMatrixDisplay(confusion_matrix=conf_matrix).plot(cmap='Blues')

# 3. ROC-AUC Curve
fpr, tpr, thresholds = roc_curve(y_va, y_pred_proba)
roc_auc = roc_auc_score(y_va, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

# **7. Report / Insights**

to be continued